In [2]:
from duckdb_assistant import DuckDBAssistant

In [3]:
dda = DuckDBAssistant()

In [4]:
dda.sync_docs()

Cloning into '/Users/ss/duckdb-assistant/chroma'...
Already on 'main'


Your branch is up to date with 'origin/main'.


'434 documents available in collection duckdb_docs'

In [5]:
resp = dda.search("Importing from a Parquet file", display_results = True)

---
layout: docu
redirect_from:
- /docs/guides/import/http_import
- /docs/guides/network_cloud_storage/http_import
- /docs/preview/guides/network_cloud_storage/http_import
- /docs/stable/guides/network_cloud_storage/http_import
title: HTTP Parquet Import
---

To load a Parquet file over HTTP(S), the [`httpfs` extension]({% link docs/current/core_extensions/httpfs/overview.md %}) is required. This can be installed using the `INSTALL` SQL command. This only needs to be run once.

```sql
INSTALL httpfs;
```

To load the `httpfs` extension for usage, use the `LOAD` SQL command:

```sql
LOAD httpfs;
```

After the `httpfs` extension is set up, Parquet files can be read over `http(s)`:

```sql
SELECT * FROM read_parquet('https://⟨domain⟩/path/to/file.parquet');
```

For example:

```sql
SELECT * FROM read_parquet('https://duckdb.org/data/prices.parquet');
```

Moreover, the `read_parquet` function itself can also be omitted thanks to DuckDB's [replacement scan mechanism]({% link docs/current/clients/c/replacement_scans.md %}):

```sql
SELECT * FROM 'https://duckdb.org/data/holdings.parquet';
```


---
layout: docu
redirect_from:
- /docs/data/parquet/tips
- /docs/preview/data/parquet/tips
- /docs/stable/data/parquet/tips
title: Parquet Tips
---

Below is a collection of tips to help when dealing with Parquet files.

## Tips for Reading Parquet Files

### Use `union_by_name` When Loading Files with Different Schemas

The `union_by_name` option can be used to unify the schema of files that have different or missing columns. For files that do not have certain columns, `NULL` values are filled in:

```sql
SELECT *
FROM read_parquet('flights*.parquet', union_by_name = true);
```

## Tips for Writing Parquet Files

Using a [glob pattern]({% link docs/current/data/multiple_files/overview.md %}#glob-syntax) upon read or a [Hive partitioning]({% link docs/current/data/partitioning/hive_partitioning.md %}) structure are good ways to transparently handle multiple files.

### Enabling `PER_THREAD_OUTPUT`

If the final number of Parquet files is not important, writing one file per thread can significantly improve performance:

```sql
COPY
    (FROM generate_series(10_000_000))
    TO 'test.parquet'
    (FORMAT parquet, PER_THREAD_OUTPUT);
```

### Selecting a `ROW_GROUP_SIZE`

The `ROW_GROUP_SIZE` parameter specifies the minimum number of rows in a Parquet row group, with a minimum value equal to DuckDB's vector size, 2,048, and a default of 122,880.
A Parquet row group is a partition of rows, consisting of a column chunk for each column in the dataset.

Compression algorithms are only applied per row group, so the larger the row group size, the more opportunities to compress the data.
On the other hand, larger row group sizes mean that each thread keeps more data in memory before flushing when streaming results.
Another argument for smaller row group sizes is that DuckDB can read Parquet row groups in parallel even within the same file and uses predicate pushdown to only scan the row groups whose metadata ranges match the `WHERE` clause of the query. However, there is some overhead associated with reading the metadata in each group.

A good rule of thumb is to ensure that the number of row groups per file is at least as large as the number of CPU threads used to query that file.
More row groups beyond the thread count would improve the speed of highly selective queries, but slow down queries that must scan the whole file like aggregations.

To write a query to a Parquet file with a different row group size, run:

```sql
COPY
    (FROM generate_series(100_000))
    TO 'row-groups.parquet'
    (FORMAT parquet, ROW_GROUP_SIZE 100_000);
```

### The `ROW_GROUPS_PER_FILE` Option

The `ROW_GROUPS_PER_FILE` parameter creates a new Parquet file if the current one has a specified number of row groups.

```sql
COPY
    (FROM generate_series(100_000))
    TO 'output-directory'
    (FORMAT parquet, ROW_GROUP_SIZE 20_000, ROW_GROUPS_PER_FILE 2);
```

> If multiple threads are active, the number of row groups in a file may slightly exceed the specified number of row groups to limit the amount of locking – similarly to the behavior of [`FILE_SIZE_BYTES`](../../sql/statements/copy#copy--to-options).
> However, if `PER_THREAD_OUTPUT` is set, only one thread writes to each file, and it becomes accurate again.

### Sorting Rows to Improve Row Group Pruning

DuckDB writes per-column statistics, including min/max values, for every row group, and as described above the reader uses these to skip row groups that cannot match a query’s `WHERE` clause.
Sorting the data on the columns that you filter on most often before writing makes these min/max ranges tight and non-overlapping between row groups, so that highly selective queries on those columns scan far fewer row groups:

```sql
COPY
    (FROM 'events.parquet' ORDER BY event_time)
    TO 'events-sorted.parquet'
    (FORMAT parquet);
```

This pairs well with a `ROW_GROUP_SIZE` that keeps each row group focused on a narrow range of the sort key.

See the [Performance Guide on “File Formats”]({% link docs/current/guides/performance/file_formats.md %}#parquet-file-sizes) for more tips.


---
layout: docu
redirect_from:
- /docs/guides/file_formats/parquet_export
- /docs/guides/import/parquet_export
- /docs/preview/guides/file_formats/parquet_export
- /docs/stable/guides/file_formats/parquet_export
title: Parquet Export
---

To export the data from a table to a Parquet file, use the `COPY` statement:

```sql
COPY tbl TO 'output.parquet' (FORMAT parquet);
```

The result of queries can also be directly exported to a Parquet file:

```sql
COPY (SELECT * FROM tbl) TO 'output.parquet' (FORMAT parquet);
```

The flags for setting compression, row group size, etc. are listed in the [Reading and Writing Parquet files]({% link docs/current/data/parquet/overview.md %}) page.


---
layout: docu
redirect_from:
- /docs/data/parquet
- /docs/data/parquet/overview
- /docs/extensions/parquet
- /docs/preview/data/parquet/overview
- /docs/stable/data/parquet/overview
title: Reading and Writing Parquet Files
---

## Examples

Read a single Parquet file:

```sql
SELECT * FROM 'test.parquet';
```

Figure out which columns/types are in a Parquet file:

```sql
DESCRIBE SELECT * FROM 'test.parquet';
```

Create a table from a Parquet file:

```sql
CREATE TABLE test AS
    SELECT * FROM 'test.parquet';
```

If the file does not end in `.parquet`, use the `read_parquet` function:

```sql
SELECT *
FROM read_parquet('test.parq');
```

Use list parameter to read three Parquet files and treat them as a single table:

```sql
SELECT *
FROM read_parquet(['file1.parquet', 'file2.parquet', 'file3.parquet']);
```

Read all files that match the glob pattern:

```sql
SELECT *
FROM 'test/*.parquet';
```

Read all files that match the glob pattern, and include the `filename` virtual column that specifies which file each row came from (this column is available by default without a configuration options since DuckDB v1.3.0):

```sql
SELECT *, filename
FROM read_parquet('test/*.parquet');
```

Use a list of globs to read all Parquet files from two specific folders:

```sql
SELECT *
FROM read_parquet(['folder1/*.parquet', 'folder2/*.parquet']);
```

Read over HTTPS:

```sql
SELECT *
FROM read_parquet('https://some.url/some_file.parquet');
```

Query the [metadata of a Parquet file]({% link docs/current/data/parquet/metadata.md %}#parquet-metadata):

```sql
SELECT *
FROM parquet_metadata('test.parquet');
```

Query the [file metadata of a Parquet file]({% link docs/current/data/parquet/metadata.md %}#parquet-file-metadata):

```sql
SELECT *
FROM parquet_file_metadata('test.parquet');
```

Query the [key-value metadata of a Parquet file]({% link docs/current/data/parquet/metadata.md %}#parquet-key-value-metadata):

```sql
SELECT *
FROM parquet_kv_metadata('test.parquet');
```

Query the [schema of a Parquet file]({% link docs/current/data/parquet/metadata.md %}#parquet-schema):

```sql
SELECT *
FROM parquet_schema('test.parquet');
```

Write the results of a query to a Parquet file using the default compression (Snappy):

```sql
COPY
    (SELECT * FROM tbl)
    TO 'result-snappy.parquet'
    (FORMAT parquet);
```

Write the results from a query to a Parquet file with specific compression and row group size:

```sql
COPY
    (FROM generate_series(100_000))
    TO 'test.parquet'
    (FORMAT parquet, COMPRESSION zstd, ROW_GROUP_SIZE 100_000);
```

Export the table contents of the entire database as parquet:

```sql
EXPORT DATABASE 'target_directory' (FORMAT parquet);
```

## Parquet Files

Parquet files are compressed columnar files that are efficient to load and process. DuckDB provides support for both reading and writing Parquet files in an efficient manner, as well as support for pushing filters and projections into the Parquet file scans.

> Parquet datasets differ based on the number of files, the size of individual files, the compression algorithm used, row group size, etc. These have a significant effect on performance. Please consult the [Performance Guide]({% link docs/current/guides/performance/file_formats.md %}) for details.

## `read_parquet` Function

| Function | Description | Example |
|:--|:--|:-----|
| `read_parquet(path_or_list_of_paths)` | Read Parquet file(s)     | `SELECT * FROM read_parquet('test.parquet');` |
| `parquet_scan(path_or_list_of_paths)` | Alias for `read_parquet` | `SELECT * FROM parquet_scan('test.parquet');` |

If your file ends in `.parquet`, the function syntax is optional. The system will automatically infer that you are reading a Parquet file:

```sql
SELECT * FROM 'test.parquet';
```

Multiple files can be read at once by providing a glob or a list of files. Refer to the [multiple files section]({% link docs/current/data/multiple_files/overview.md %}) for more information.

### Parameters

There are a number of options exposed that can be passed to the `read_parquet` function or the [`COPY` statement]({% link docs/current/sql/statements/copy.md %}).

| Name | Description | Type | Default |
|:--|:-----|:-|:-|
| `binary_as_string` | Parquet files generated by legacy writers do not correctly set the `UTF8` flag for strings, causing string columns to be loaded as `BLOB` instead. Set this to true to load binary columns as strings. | `BOOL` | `false` |
| `can_have_nan` | Whether `FLOAT` and `DOUBLE` columns may contain `NaN` values. When set to true, the reader accounts for `NaN` when using a column's min/max statistics for filter pushdown, since `NaN` does not compare as ordered. | `BOOL` | `false` |
| `encryption_config` | Configuration for [Parquet encryption]({% link docs/current/data/parquet/encryption.md %}). | `STRUCT` | - |
| `filename` | Whether or not an extra `filename` column should be included in the result. Since DuckDB v1.3.0, the `filename` column is added automatically as a virtual column and this option is only kept for compatibility reasons. | `BOOL` | `false` |
| `file_row_number` | Whether or not to include the `file_row_number` column. | `BOOL` | `false` |
| `hive_partitioning` | Whether or not to interpret the path as a [Hive partitioned path]({% link docs/current/data/partitioning/hive_partitioning.md %}). | `BOOL` | (auto-detected) |
| `union_by_name` | Whether the columns of multiple schemas should be [unified by name]({% link docs/current/data/multiple_files/combining_schemas.md %}), rather than by position. | `BOOL` | `false` |
| `schema` | Allows you to read a Parquet file as if it has the supplied schema. Field IDs are required. | `MAP` | `NULL` |

## Using the `schema` Parameter

The `schema` parameter allows you to read the Parquet file using a specific schema. This is useful for renaming, adding, deleting, reordering, or casting columns when reading Parquet files.

To use the `schema` parameter, field IDs are required. To make them available when creating the Parquet using DuckDB, use:

```sql
COPY (SELECT 42::INTEGER AS i) TO 'integers.parquet' (FIELD_IDS {i: 0});
```

Reading Parquet files:

```sql
SELECT *
FROM read_parquet('integers.parquet', schema = MAP {
                    0: {name: 'renamed_i', type: 'BIGINT', default_value: NULL},
                    1: {name: 'new_column', type: 'UTINYINT', default_value: 43}
                  });
```

```text
┌───────────┬────────────┐
│ renamed_i │ new_column │
│   int64   │   uint8    │
├───────────┼────────────┤
│        42 │         43 │
└───────────┴────────────┘
```

> The `schema` parameter cannot be combined with `union_by_name = true`.

## Partial Reading

DuckDB supports projection pushdown into the Parquet file itself. That is to say, when querying a Parquet file, only the columns required for the query are read. This allows you to read only the part of the Parquet file that you are interested in. This will be done automatically by DuckDB.

DuckDB also supports filter pushdown into the Parquet reader. When you apply a filter to a column that is scanned from a Parquet file, the filter will be pushed down into the scan, and can even be used to skip parts of the file using the built-in zonemaps. Note that this will depend on whether or not your Parquet file contains zonemaps.

Filter and projection pushdown provide significant performance benefits. See [our blog post “Querying Parquet with Precision Using DuckDB”]({% post_url 2021-06-25-querying-parquet %}) for more information.

## Inserts and Views

You can also insert the data into a table or create a table from the Parquet file directly. This will load the data from the Parquet file and insert it into the database:

Insert the data from the Parquet file in the table:

```sql
INSERT INTO people
    SELECT * FROM read_parquet('test.parquet');
```

Create a table directly from a Parquet file:

```sql
CREATE TABLE people AS
    SELECT * FROM read_parquet('test.parquet');
```

If you wish to keep the data stored inside the Parquet file, but want to query the Parquet file directly, you can create a view over the `read_parquet` function. You can then query the Parquet file as if it were a built-in table:

Create a view over the Parquet file:

```sql
CREATE VIEW people AS
    SELECT * FROM read_parquet('test.parquet');
```

Query the Parquet file:

```sql
SELECT * FROM people;
```

## Writing to Parquet Files

DuckDB also has support for writing to Parquet files using the `COPY` statement syntax. See the [`COPY` Statement page]({% link docs/current/sql/statements/copy.md %}) for details, including all possible parameters for the `COPY` statement.

Write a query to a Snappy-compressed Parquet file:

```sql
COPY
    (SELECT * FROM tbl)
    TO 'result-snappy.parquet'
    (FORMAT parquet);
```

Write `tbl` to a zstd-compressed Parquet file:

```sql
COPY tbl
    TO 'result-zstd.parquet'
    (FORMAT parquet, COMPRESSION zstd);
```

Write `tbl` to a zstd-compressed Parquet file with the lowest compression level yielding the fastest compression:

```sql
COPY tbl
    TO 'result-zstd.parquet'
    (FORMAT parquet, COMPRESSION zstd, COMPRESSION_LEVEL 1);
```

Write to Parquet file with [key-value metadata]({% link docs/current/data/parquet/metadata.md %}#parquet-key-value-metadata):

```sql
COPY (
    SELECT
        42 AS number,
        true AS is_even
) TO 'kv_metadata.parquet' (
    FORMAT parquet,
    KV_METADATA {
        number: 'Answer to life, universe, and everything',
        is_even: 'not ''odd''' -- single quotes in values must be escaped
    }
);
```

Write to a Parquet v2 file:

```sql
COPY tbl
    TO 'result-v2.parquet'
    (FORMAT parquet, PARQUET_VERSION 'V2');
```

Write a CSV file to an uncompressed Parquet file:

```sql
COPY
    'test.csv'
    TO 'result-uncompressed.parquet'
    (FORMAT parquet, COMPRESSION uncompressed);
```

Write a query to a Parquet file with zstd-compression and row group size:

```sql
COPY
    (FROM generate_series(100_000))
    TO 'row-groups-zstd.parquet'
    (FORMAT parquet, COMPRESSION zstd, ROW_GROUP_SIZE 100_000);
```

Write data to an LZ4-compressed Parquet file:

```sql
COPY
    (FROM generate_series(100_000))
    TO 'result-lz4.parquet'
    (FORMAT parquet, COMPRESSION lz4);
```

Or, equivalently:

```sql
COPY
    (FROM generate_series(100_000))
    TO 'result-lz4.parquet'
    (FORMAT parquet, COMPRESSION lz4_raw);
```

Write data to a Brotli-compressed Parquet file:

```sql
COPY
    (FROM generate_series(100_000))
    TO 'result-brotli.parquet'
    (FORMAT parquet, COMPRESSION brotli);
```

To configure the page size of Parquet file's dictionary pages, use the `STRING_DICTIONARY_PAGE_SIZE_LIMIT` option (default: 1 MB):

```sql
COPY
    lineitem
    TO 'lineitem-with-custom-dictionary-size.parquet'
    (FORMAT parquet, STRING_DICTIONARY_PAGE_SIZE_LIMIT 100_000);
```

DuckDB's `EXPORT` command can be used to export an entire database to a series of Parquet files. See the [“`EXPORT` statement” page]({% link docs/current/sql/statements/export.md %}) for more details:

Export the table contents of the entire database as Parquet:

```sql
EXPORT DATABASE 'target_directory' (FORMAT parquet);
```

## Encryption

DuckDB supports reading and writing [encrypted Parquet files]({% link docs/current/data/parquet/encryption.md %}).

## Supported Features

The list of supported Parquet features is available in the [Parquet documentation's “Implementation status” page](https://parquet.apache.org/docs/file-format/implementationstatus/).

## Installing and Loading the Parquet Extension

The support for Parquet files is enabled via extension. The `parquet` extension is bundled with almost all clients. However, if your client does not bundle the `parquet` extension, the extension must be installed separately:

```sql
INSTALL parquet;
```


---
layout: docu
redirect_from:
- /docs/guides/file_formats/query_parquet
- /docs/guides/import/query_parquet
- /docs/preview/guides/file_formats/query_parquet
- /docs/stable/guides/file_formats/query_parquet
title: Querying Parquet Files
---

To run a query directly on a Parquet file, use the `read_parquet` function in the `FROM` clause of a query.

```sql
SELECT * FROM read_parquet('input.parquet');
```

The Parquet file will be processed in parallel. Filters will be automatically pushed down into the Parquet scan, and only the relevant columns will be read automatically.

For more information see the blog post [“Querying Parquet with Precision using DuckDB”]({% post_url 2021-06-25-querying-parquet %}).


In [7]:
resp

{'documents': ["---\nlayout: docu\nredirect_from:\n- /docs/guides/import/http_import\n- /docs/guides/network_cloud_storage/http_import\n- /docs/preview/guides/network_cloud_storage/http_import\n- /docs/stable/guides/network_cloud_storage/http_import\ntitle: HTTP Parquet Import\n---\n\nTo load a Parquet file over HTTP(S), the [`httpfs` extension]({% link docs/current/core_extensions/httpfs/overview.md %}) is required. This can be installed using the `INSTALL` SQL command. This only needs to be run once.\n\n```sql\nINSTALL httpfs;\n```\n\nTo load the `httpfs` extension for usage, use the `LOAD` SQL command:\n\n```sql\nLOAD httpfs;\n```\n\nAfter the `httpfs` extension is set up, Parquet files can be read over `http(s)`:\n\n```sql\nSELECT * FROM read_parquet('https://⟨domain⟩/path/to/file.parquet');\n```\n\nFor example:\n\n```sql\nSELECT * FROM read_parquet('https://duckdb.org/data/prices.parquet');\n```\n\nMoreover, the `read_parquet` function itself can also be omitted thanks to DuckDB's

In [22]:
resp2 = display(Markdown(resp["documents"][2]))

---
layout: docu
redirect_from:
- /docs/guides/file_formats/parquet_export
- /docs/guides/import/parquet_export
- /docs/preview/guides/file_formats/parquet_export
- /docs/stable/guides/file_formats/parquet_export
title: Parquet Export
---

To export the data from a table to a Parquet file, use the `COPY` statement:

```sql
COPY tbl TO 'output.parquet' (FORMAT parquet);
```

The result of queries can also be directly exported to a Parquet file:

```sql
COPY (SELECT * FROM tbl) TO 'output.parquet' (FORMAT parquet);
```

The flags for setting compression, row group size, etc. are listed in the [Reading and Writing Parquet files]({% link docs/current/data/parquet/overview.md %}) page.


None
